# Fast R-CNN

**Paper**: Girshick, ICCV 2015

## The Problem with R-CNN

R-CNN's bottleneck: **2000 separate CNN forward passes** per image. Each proposal is cropped, warped, and forwarded independently — most computation is redundant because proposals heavily overlap.

## Fast R-CNN's Key Insight: Process the Image Once

<img src='../figures/fast_rcnn_pipeline.png' width='700'/>

1. Run the **full image through the CNN once** → shared feature map
2. For each region proposal: project it onto the feature map, extract with **RoI Pooling**
3. Pass each RoI feature through FC layers → **jointly** predict class + box offsets

**Result**: Shared feature computation across all proposals. 9x faster than R-CNN at inference.

## What is RoI Pooling?

<img src='../figures/roi_pooling.png' width='600'/>

RoI Pooling maps a variable-size region of the feature map to a **fixed-size output** (e.g., 7x7) using max-pooling within a grid of sub-windows. This allows FC layers to accept arbitrary region sizes.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torchvision.ops import roi_pool, roi_align
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import requests
from io import BytesIO

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## RoI Pooling in PyTorch

PyTorch provides `torchvision.ops.roi_pool` and the improved `roi_align`. `roi_align` uses bilinear interpolation instead of hard quantization — this avoids the misalignment artifacts that hurt Mask R-CNN accuracy.

In [ ]:
# Simulate a feature map and some proposals
B, C, H, W = 1, 512, 32, 32  # feature map after backbone
feat_map = torch.randn(B, C, H, W)

# Region proposals as [batch_idx, x1, y1, x2, y2] in feature-map coordinates
# (the projection from image coords divides by stride)
stride = 8  # e.g., VGG16 with 3 max-pool layers
proposals_img = torch.tensor([
    [0,  50., 80., 180., 200.],  # proposal 1 in image coords
    [0, 100., 20., 220.,  90.],  # proposal 2
    [0,  10., 10.,  60.,  60.],  # proposal 3
], dtype=torch.float32)
proposals_feat = proposals_img.clone()
proposals_feat[:, 1:] /= stride  # project to feature map coords

# RoI Pooling → fixed 7x7 output per region
roi_out = roi_pool(feat_map, proposals_feat, output_size=(7,7), spatial_scale=1.0)
print(f'Feature map:      {tuple(feat_map.shape)}')
print(f'Proposals:        {proposals_feat.shape[0]} boxes')
print(f'RoI Pool output:  {tuple(roi_out.shape)}  (3 regions x 512 x 7 x 7)')

# RoI Align (better, used in Faster R-CNN / Mask R-CNN)
roi_align_out = roi_align(feat_map, proposals_feat, output_size=(7,7), spatial_scale=1.0)
print(f'RoI Align output: {tuple(roi_align_out.shape)}')
print('Each 512x7x7 feature is flattened and passed to FC layers for cls+box prediction.')

## Fast R-CNN Head

After RoI Pooling, each region's features go through two FC layers, then two parallel branches:
- **Classifier**: softmax over (N_classes + 1) — includes background class
- **Box regressor**: 4 * N_classes offsets (one box per class)

In [ ]:
class FastRCNNHead(nn.Module):
    def __init__(self, in_channels=512, roi_size=7, n_classes=21):
        super().__init__()
        flat_dim = in_channels * roi_size * roi_size
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_dim, 4096), nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),     nn.ReLU(),
            nn.Dropout(0.5),
        )
        self.cls_head = nn.Linear(4096, n_classes)        # class scores
        self.box_head = nn.Linear(4096, n_classes * 4)    # box offsets per class

    def forward(self, roi_features):
        feat   = self.fc(roi_features)
        scores = self.cls_head(feat)                  # (N, n_classes)
        deltas = self.box_head(feat)                  # (N, n_classes*4)
        return scores, deltas


head = FastRCNNHead(in_channels=512, roi_size=7, n_classes=21)
scores, deltas = head(roi_out)
print(f'RoI features in: {tuple(roi_out.shape)}')
print(f'Class scores:    {tuple(scores.shape)}  (3 proposals x 21 classes)')
print(f'Box offsets:     {tuple(deltas.shape)}  (3 proposals x 21*4 offsets)')

## Fast R-CNN Loss

Fast R-CNN trains end-to-end with a **multi-task loss**:

$$L = L_{cls} + \lambda \cdot \mathbb{1}[u \geq 1] \cdot L_{loc}$$

- $L_{cls}$: cross-entropy over all classes (including background)
- $L_{loc}$: smooth-L1 loss on box offsets **only for non-background proposals**
- $\lambda=1$, $\mathbb{1}[u \geq 1]$ ignores background proposals in box loss

**Smooth-L1** (Huber loss) is more robust to outliers than L2 for box regression.

In [ ]:
def smooth_l1_loss(pred, target, beta=1.0):
    diff = (pred - target).abs()
    loss = torch.where(diff < beta, 0.5 * diff**2 / beta, diff - 0.5*beta)
    return loss.mean()

def fast_rcnn_loss(scores, deltas, gt_labels, gt_deltas, lam=1.0):
    # Classification loss
    cls_loss = F.cross_entropy(scores, gt_labels)

    # Box regression loss — only for foreground proposals (label >= 1)
    fg_mask = gt_labels >= 1
    if fg_mask.sum() > 0:
        n_cls = scores.shape[1]
        # Select predicted offsets for the ground-truth class
        N = fg_mask.sum()
        fg_labels  = gt_labels[fg_mask]
        fg_deltas  = deltas[fg_mask]
        # Index [batch, class*4 : class*4+4]
        idx = fg_labels.unsqueeze(1) * 4 + torch.arange(4, device=deltas.device)
        pred_fg = fg_deltas.gather(1, idx)
        box_loss = smooth_l1_loss(pred_fg, gt_deltas[fg_mask])
    else:
        box_loss = torch.tensor(0.0)

    return cls_loss + lam * box_loss, cls_loss.item(), box_loss.item()


# Demo with dummy targets
N_rois    = 3
n_classes = 21
gt_labels = torch.tensor([0, 5, 0])           # 0=background, 5=foreground class
gt_deltas = torch.randn(N_rois, 4)            # regression targets
scores_d  = torch.randn(N_rois, n_classes)
deltas_d  = torch.randn(N_rois, n_classes*4)

total, cls, box = fast_rcnn_loss(scores_d, deltas_d, gt_labels, gt_deltas)
print(f'Total loss: {total.item():.4f}')
print(f'  cls_loss: {cls:.4f}')
print(f'  box_loss: {box:.4f}  (only for fg proposals)')

## Using torchvision's Pre-built Fast/Faster R-CNN

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights

COCO_NAMES = [
    '__background__','person','bicycle','car','motorcycle','airplane','bus','train',
    'truck','boat','traffic light','fire hydrant','stop sign','parking meter','bench',
    'bird','cat','dog','horse','sheep','cow','elephant','bear','zebra','giraffe',
    'backpack','umbrella','handbag','tie','suitcase','frisbee','skis','snowboard',
    'sports ball','kite','baseball bat','baseball glove','skateboard','surfboard',
    'tennis racket','bottle','wine glass','cup','fork','knife','spoon','bowl',
    'banana','apple','sandwich','orange','broccoli','carrot','hot dog','pizza',
    'donut','cake','chair','couch','potted plant','bed','dining table','toilet',
    'tv','laptop','mouse','remote','keyboard','cell phone','microwave','oven',
    'toaster','sink','refrigerator','book','clock','vase','scissors','teddy bear',
    'hair drier','toothbrush'
]

model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
model = model.to(device).eval()

try:
    img_pil = Image.open(BytesIO(requests.get('https://ultralytics.com/images/zidane.jpg',timeout=8).content)).convert('RGB')
except Exception:
    img_pil = Image.fromarray(np.random.randint(0,255,(480,640,3),dtype=np.uint8))

tf = T.Compose([T.ToTensor()])
img_tensor = tf(img_pil).to(device)

with torch.no_grad():
    predictions = model([img_tensor])

pred = predictions[0]
keep = pred['scores'] > 0.5
boxes  = pred['boxes'][keep].cpu().numpy()
labels = pred['labels'][keep].cpu().numpy()
scores = pred['scores'][keep].cpu().numpy()

fig, ax = plt.subplots(1, 1, figsize=(12,7))
ax.imshow(img_pil)
colors = plt.cm.Set1(np.linspace(0,1,len(boxes)))
for box,label,score,c in zip(boxes,labels,scores,colors):
    x1,y1,x2,y2 = box
    ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,lw=2,edgecolor=c,facecolor='none'))
    ax.text(x1,y1-4,f'{COCO_NAMES[label]} {score:.2f}',color=c,fontsize=9,fontweight='bold')
ax.set_title(f'Faster R-CNN (ResNet50+FPN) — {len(boxes)} detections @ score>0.5')
ax.axis('off'); plt.tight_layout(); plt.show()

## Fast R-CNN vs R-CNN

| | R-CNN | Fast R-CNN |
|---|---|---|
| **CNN forward passes** | 2000 (one per proposal) | 1 (full image) + RoI Pool per proposal |
| **Training** | 3 separate stages (CNN, SVM, regressor) | End-to-end single loss |
| **Inference time** | ~47 sec/image | ~2.3 sec/image |
| **mAP VOC07** | 66.0% | 70.0% |
| **Remaining bottleneck** | — | Selective Search (~2 sec) still dominates |

Fast R-CNN's remaining bottleneck was **Selective Search** — a CPU-based algorithm that couldn't be parallelized with GPU inference. This motivated Faster R-CNN's Region Proposal Network.